# Lab 22 — DPO/ORPO Alignment: T4 Executed Core Submission

> Preserved from the actual Colab T4 run on 2026-08-24. This submission copy contains setup + NB1–NB4 only; optional NB5/NB6 were not claimed.


---
## ⚙️ Section 0: Secrets, Environment & Google Drive


In [ ]:
# 0a. Load secrets/config từ Colab Secrets; không bao giờ in giá trị key
import os
SECRET_KEYS = ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'HF_TOKEN', 'WANDB_API_KEY']
CONFIG_KEYS = ['OPENAI_JUDGE_MODEL', 'OPENAI_CROSS_JUDGE_MODEL',
               'ANTHROPIC_JUDGE_MODEL', 'JUDGE_MODEL',
               'HF_REPO', 'HF_GGUF_REPO', 'WANDB_PROJECT',
               'GITHUB_USER', 'GITHUB_REPO', 'DRIVE_OUT', 'BACKUP_GGUF_TO_DRIVE',
               'RUN_EXHAUSTIVE_BONUS']
try:
    from google.colab import userdata
    for key in SECRET_KEYS + CONFIG_KEYS:
        try:
            value = userdata.get(key)
        except Exception:
            value = None
        if value:
            os.environ[key] = value
    print('Colab Secrets loaded: ' + ', '.join(k for k in SECRET_KEYS if os.environ.get(k)))
except ImportError:
    print('Not on Colab — will load local .env after the repo is available')


In [ ]:
# 0b. Mount Google Drive để backup artifacts
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUT = os.environ.get('DRIVE_OUT', '/content/drive/MyDrive/Lab22_DPO_artifacts')
    os.makedirs(DRIVE_OUT, exist_ok=True)
    os.environ['DRIVE_OUT'] = DRIVE_OUT
    print(f'✓ Google Drive mounted → {DRIVE_OUT}')
except Exception as e:
    print(f'Drive mount skipped: {e}')
    DRIVE_OUT = None
    os.environ.pop('DRIVE_OUT', None)


In [ ]:
# 0c. Set COMPUTE_TIER, probe GPU
import os, torch
os.environ.setdefault('COMPUTE_TIER', 'T4')
# Lab coach recommended dataset (higher quality native VN, same Alpaca format)
os.environ.setdefault('SFT_DATASET', 'bkai-foundation-models/vi-alpaca')
assert torch.cuda.is_available(), 'Enable GPU: Runtime → Change runtime type → T4 GPU'
gpu = torch.cuda.get_device_properties(0)
VRAM_GB = gpu.total_memory / 1e9
print(f'GPU: {gpu.name}  ({VRAM_GB:.1f} GB)')
if VRAM_GB >= 35:
    os.environ['COMPUTE_TIER'] = 'BIGGPU'
    print('→ Auto-detected BigGPU tier (A100/L4)')
else:
    print('→ T4 tier confirmed')
TIER = os.environ['COMPUTE_TIER']


In [ ]:
# 0d. Screenshot: GPU info (capture output as 01-setup-gpu.png)
import os
from pathlib import Path
WORK = Path('/content/lab22')
print('Ready to clone repo')


In [21]:
# 0e. Clone đúng repo đã cấu hình
GITHUB_USER = os.environ.get('GITHUB_USER', 'ZukaNoPro2k5')
REPO_NAME = os.environ.get('GITHUB_REPO', 'K4-Track3-Day22-DPO-ORPO-Alignment')
import subprocess, os
# Always leave WORK before Git commands. Reusing it preserves expensive artifacts
# on a repeated Run All and avoids deleting the subprocess' current directory.
os.chdir('/content')
if (WORK / '.git').exists():
    result = subprocess.run(['git', '-C', str(WORK), 'pull', '--ff-only'], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'Workspace update failed: {result.stderr}')
    print(f'✓ Reusing existing workspace at {WORK}')
else:
    result = subprocess.run(
        ['git', 'clone', f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git', str(WORK)],
        cwd='/content', capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f'Clone failed: {result.stderr}')
    print(f'✓ Cloned {GITHUB_USER}/{REPO_NAME} to {WORK}')
for d in ['submission/screenshots', 'adapters/sft-mini', 'adapters/dpo', 'gguf']:
    (WORK / d).mkdir(parents=True, exist_ok=True)
os.chdir(WORK / 'notebooks')


✓ Reusing existing workspace at /content/lab22


In [22]:
# 0f. Install dependencies từ source-of-truth requirements.txt (3-8 min)
import subprocess, sys
probe = subprocess.run([sys.executable, '-c', 'import unsloth, torch, trl, peft, datasets, bitsandbytes'],
                       capture_output=True, text=True)
if probe.returncode == 0:
    print('✓ Dependencies already available — skipping pip install')
else:
    print('Installing dependencies... (~3-5 min)')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    str(WORK / 'requirements.txt')], check=True)
    print('✓ All packages installed')


✓ Dependencies already available — skipping pip install


In [23]:
# 0g. Load optional repo .env, validate config, and define fail-fast runner
import os, subprocess, sys
from dotenv import load_dotenv
load_dotenv(WORK / '.env', override=False)  # local/server-mounted file only
os.environ.setdefault('SFT_DATASET', 'bkai-foundation-models/vi-alpaca')
os.environ.setdefault('PREF_DATASET', 'argilla/ultrafeedback-binarized-preferences-cleaned')
os.environ.setdefault('WANDB_PROJECT', 'lab22-dpo')
if os.environ.get('WANDB_API_KEY'):
    os.environ.setdefault('WANDB_MODE', 'online')
run_exhaustive = os.environ.get('RUN_EXHAUSTIVE_BONUS', '1').lower() not in {'0', 'false', 'no'}
if run_exhaustive:
    required_bonus_keys = ['OPENAI_API_KEY', 'HF_TOKEN', 'WANDB_API_KEY']
    missing = [key for key in required_bonus_keys if not os.environ.get(key)]
    # VS Code's Colab extension can fail to propagate browser Secret access.
    # Offer an in-memory fallback; getpass never prints or writes the values.
    if missing:
        from getpass import getpass
        print('Colab Secrets were not exposed to this kernel. Enter missing values for this runtime only.')
        for key in missing:
            value = getpass(f'{key}: ')
            if value:
                os.environ[key] = value.strip()
        missing = [key for key in required_bonus_keys if not os.environ.get(key)]
    if missing:
        raise RuntimeError('Exhaustive bonus mode needs Colab Secrets: ' + ', '.join(missing))
    if not os.environ.get('ANTHROPIC_API_KEY'):
        print('ⓘ ANTHROPIC_API_KEY absent — running every available bonus; only cross-judge +4 is skipped.')

def run_step(label, *args, check=True):
    import re
    print(f'\n=== {label} ===', flush=True)
    log_dir = WORK / 'data/eval/logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / (re.sub(r'[^a-z0-9]+', '-', label.lower()).strip('-') + '.log')
    lines = []
    with log_path.open('w', encoding='utf-8') as log_file:
        proc = subprocess.Popen([sys.executable, *args], cwd=WORK, env=os.environ.copy(),
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True)
            log_file.write(line)
            lines.append(line)
        returncode = proc.wait()
    if check and returncode != 0:
        print(f'\n--- {label}: final 80 log lines ({log_path}) ---')
        print(''.join(lines[-80:]))
        raise RuntimeError(f'{label} failed with exit code {returncode}; see {log_path}')
    return returncode

print(f"SFT_DATASET={os.environ['SFT_DATASET']}")
print(f"PREF_DATASET={os.environ['PREF_DATASET']}")
print('Enabled integrations: ' + (', '.join(k for k in SECRET_KEYS if os.environ.get(k)) or 'none'))

# Fail early on an incompatible environment instead of after training starts.
import shutil
free_gb = shutil.disk_usage('/content').free / 1024**3
assert free_gb >= 20, f'Need at least 20 GB free disk; only {free_gb:.1f} GB available'
preflight_code = (
    "import unsloth, torch, trl, peft, datasets, bitsandbytes; "
    "assert torch.cuda.is_available(), 'CUDA unavailable'; "
    "print(f'Preflight OK: torch={torch.__version__}, trl={trl.__version__}, peft={peft.__version__}')"
)
subprocess.run([sys.executable, '-c', preflight_code], cwd=WORK, check=True, env=os.environ.copy())
print(f'Free disk: {free_gb:.1f} GB')


ⓘ ANTHROPIC_API_KEY absent — running every available bonus; only cross-judge +4 is skipped.
SFT_DATASET=bkai-foundation-models/vi-alpaca
PREF_DATASET=argilla/ultrafeedback-binarized-preferences-cleaned
Enabled integrations: OPENAI_API_KEY, HF_TOKEN, WANDB_API_KEY
Free disk: 56.9 GB


In [25]:
!git -C /content/lab22 pull --ff-only

remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 14 (delta 11), reused 14 (delta 11), pack-reused 0 (from 0)
Unpacking objects: 100% (14/14), 2.93 KiB | 428.00 KiB/s, done.
From https://github.com/ZukaNoPro2k5/K4-Track3-Day22-DPO-ORPO-Alignment
   ab93004..37f7331  main       -> origin/main
Updating ab93004..37f7331
Fast-forward
 bonus/demo/serve.py                | 6 ++++++
 bonus/train.py                     | 6 ++++++
 notebooks/01_sft_mini.py           | 4 ++++
 notebooks/02_preference_data.py    | 3 +++
 notebooks/04_compare_and_eval.py   | 3 +++
 notebooks/06_benchmark.py          | 3 +++
 scripts/eval_beta_sweep.py         | 3 +++
 scripts/prepare_preference_data.py | 3 +++
 8 files changed, 31 insertions(+)


---
## 🤖 NB1 — SFT-mini Build (~10 min T4)


In [26]:
# NB1: Run notebook 1 — SFT mini checkpoint
import os
os.environ.setdefault('COMPUTE_TIER', 'T4')
os.environ.setdefault('SFT_DATASET', 'bkai-foundation-models/vi-alpaca')

run_step('NB1 — SFT mini', 'notebooks/01_sft_mini.py')



=== NB1 — SFT mini ===


2026-08-24 13:34:20.337497: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
COMPUTE_TIER:    T4
BASE_MODEL:      unsloth/Qwen2.5-3B-bnb-4bit
SFT_DATASET:     bkai-foundation-models/vi-alpaca  (slice: 1000)
max_seq_length:  512
effective batch: 8
output:          /content/lab22/adapters/sft-mini
GPU: Tesla T4  (15.6 GB)
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = Fal

0

In [27]:
# NB1 sanity check: verify adapter exists
from pathlib import Path
WORK = Path('/content/lab22')
adapter_cfg = WORK / 'adapters/sft-mini/adapter_config.json'
assert adapter_cfg.exists(), f'Missing {adapter_cfg}'
import json
cfg = json.loads(adapter_cfg.read_text())
assert cfg.get('r') == 16, f'r={cfg.get("r")} != 16'
assert cfg.get('lora_alpha') == 32, f'lora_alpha={cfg.get("lora_alpha")} != 32'
print(f'✓ NB1 check: adapter_config.json verified (r={cfg["r"]}, lora_alpha={cfg["lora_alpha"]})')


✓ NB1 check: adapter_config.json verified (r=16, lora_alpha=32)


---
## 📊 NB2 — Preference Data Prep (~2 min)


In [28]:
# NB2: Run notebook 2 — preference data
run_step('NB2 — preference data', 'notebooks/02_preference_data.py')



=== NB2 — preference data ===
COMPUTE_TIER:    T4
PREF_DATASET:    argilla/ultrafeedback-binarized-preferences-cleaned  (slice: 1000)
MAX_LEN:         512
MAX_PROMPT_LEN:  256
output:          /content/lab22/data/pref
Tokenizer: Qwen2TokenizerFast  vocab=151,643

Generating train split: 100%|██████████| 60917/60917 [00:01<00:00, 49071.82 examples/s]
Loaded 1000 pairs. Columns: ['source', 'prompt', 'chosen', 'chosen-rating', 'chosen-model', 'rejected', 'rejected-rating', 'rejected-model']

Map: 100%|██████████| 1000/1000 [00:00<00:00, 3513.19 examples/s]
Formatted: 1000 pairs · cols: ['prompt', 'chosen', 'rejected']

────── Example 1 ──────
PROMPT (133 tok):
<|im_start|>system You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|> <|im_start|>user Can you write a C++ program that prompts the user to enter the name of a [...]

CHOSEN (488 tok):
Here's a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea

0

---
## 🎯 NB3 — DPO Training (~15-20 min T4)


In [29]:
# NB3: Run notebook 3 — DPO training
run_step('NB3 — DPO training', 'notebooks/03_dpo_train.py')

import json
try:
    metrics = json.loads(open('/content/lab22/adapters/dpo/dpo_metrics.json').read())
    print(f'✓ NB3 DONE  loss={metrics["final_train_loss"]:.4f}  gap={metrics.get("end_reward_gap","N/A")}')
except Exception as e:
    print('⚠ Could not read DPO metrics, maybe training failed?')



=== NB3 — DPO training ===
2026-08-24 13:43:55.676401: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
COMPUTE_TIER:    T4
BASE_MODEL:      unsloth/Qwen2.5-3B-bnb-4bit
DPO hyperparams: beta=0.1  lr=5e-07  epochs=1
max_length:      512  (prompt=256)
effective batch: 8
SFT input:       /content/lab22/adapters/sft-mini
output:          /content/lab22/adapters/dpo
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat1

---
## 📋 NB4 — Side-by-Side Eval + Judge (~3-5 min)


In [30]:
# NB4: Run notebook 4 — compare and eval
run_step('NB4 — compare and judge', 'notebooks/04_compare_and_eval.py')

import json
from collections import Counter
from pathlib import Path
judge_f = Path('/content/lab22/data/eval/judge_results.json')
if judge_f.exists():
    wins = Counter(x.get('winner') for x in json.loads(judge_f.read_text()))
    print(f'✓ NB4 DONE  SFT={wins.get("A",0)}/8  DPO={wins.get("B",0)}/8  tie={wins.get("tie",0)}/8')



=== NB4 — compare and judge ===
2026-08-24 14:06:11.892971: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Saved 8 eval prompts to /content/lab22/data/eval/prompts.json
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Generating with SFT-only adapter...
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore 